# Lesson 20 — Enterprise Capstone

## What you will learn
- Integrate ALL enterprise pillars in one system:
  - **L16**: Async + multi-tenant thread IDs
  - **L17**: JWT auth + RBAC gate
  - **L18**: Structured logging + metrics
  - **L19**: Event-driven triggering
- **Supervisor** routing: db_agent, report_agent, hr_agent
- **Cost governance**: reject requests over token budget
- End-to-end walkthrough from HTTP request → agent response

## Full system architecture
```
HTTP/Event
   ↓
[JWT Auth + RBAC]          L17
   ↓
[Observability wrapper]    L18
   ↓
[Supervisor]               L16 (async, multi-tenant)
   ↓         ↓        ↓
[db_agent] [report] [hr]   L6 / L10 patterns
   ↓
[Cost governor]            budget check
   ↓
Response
```

In [ ]:
# !pip install langgraph langchain-ollama pyjwt

## Step 1 — Enterprise State

In [ ]:
import jwt, time, uuid, json, logging
from typing import Annotated
from typing_extensions import TypedDict
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

llm = ChatOllama(model="llama3", temperature=0)
JWT_SECRET = "enterprise-capstone-secret"

def merge_metrics(a: dict, b: dict) -> dict:
    m = dict(a)
    for k, v in b.items():
        m[k] = m.get(k, 0) + v if isinstance(v, (int, float)) else v
    return m

class EnterpriseState(TypedDict):
    messages:   Annotated[list, add_messages]
    # Auth context (L17)
    user_id:    str
    tenant_id:  str
    role:       str
    # Routing (L16)
    next_agent: str
    # Observability (L18)
    trace_id:   str
    metrics:    Annotated[dict, merge_metrics]
    # Cost governance
    token_budget: int
    tokens_used:  int
    # Output
    response:   str
    error:      str

print("Enterprise state defined")

## Step 2 — Auth Gate (L17)

In [ ]:
ROLE_PERMS = {
    "user":    {"db_agent"},
    "manager": {"db_agent", "report_agent"},
    "admin":   {"db_agent", "report_agent", "hr_agent"},
}

def auth_gate_node(state: EnterpriseState) -> dict:
    t0 = time.perf_counter()
    question = state["messages"][-1].content.lower() if state["messages"] else ""
    
    # Determine target agent
    if any(w in question for w in ["report", "weekly", "summary"]):
        target = "report_agent"
    elif any(w in question for w in ["hr", "hire", "salary", "employee"]):
        target = "hr_agent"
    else:
        target = "db_agent"
    
    # RBAC check
    allowed = ROLE_PERMS.get(state["role"], set())
    if target not in allowed:
        return {
            "error": f"Role '{state['role']}' cannot access '{target}'",
            "next_agent": "denied",
            "metrics": {"auth_denied": 1, "auth_ms": round((time.perf_counter()-t0)*1000, 1)},
        }
    
    return {
        "next_agent": target,
        "metrics": {"auth_ok": 1, "auth_ms": round((time.perf_counter()-t0)*1000, 1)},
    }

print("Auth gate ready")

## Step 3 — Cost Governor

In [ ]:
def cost_governor_node(state: EnterpriseState) -> dict:
    """Reject if over token budget."""
    # Rough estimate: 4 chars ≈ 1 token
    total_chars = sum(len(getattr(m, "content", "")) for m in state["messages"])
    estimated_tokens = total_chars // 4
    
    if estimated_tokens > state["token_budget"]:
        return {
            "error": f"Token budget exceeded: {estimated_tokens} > {state['token_budget']}",
            "next_agent": "budget_exceeded",
            "tokens_used": estimated_tokens,
            "metrics": {"budget_exceeded": 1},
        }
    return {"tokens_used": estimated_tokens, "metrics": {"tokens_used": estimated_tokens}}

print("Cost governor ready")

## Step 4 — Specialist Agents

In [ ]:
def db_agent_node(state: EnterpriseState) -> dict:
    t0 = time.perf_counter()
    response = llm.invoke([
        SystemMessage(content="You are a database analyst. Answer data questions concisely."),
        *state["messages"]
    ])
    return {
        "messages": [response],
        "response": response.content,
        "metrics": {"db_agent_calls": 1, "db_agent_ms": round((time.perf_counter()-t0)*1000, 1)},
    }

def report_agent_node(state: EnterpriseState) -> dict:
    t0 = time.perf_counter()
    response = llm.invoke([
        SystemMessage(content="You are a report writer. Generate concise business reports."),
        *state["messages"]
    ])
    return {
        "messages": [response],
        "response": response.content,
        "metrics": {"report_agent_calls": 1, "report_agent_ms": round((time.perf_counter()-t0)*1000, 1)},
    }

def hr_agent_node(state: EnterpriseState) -> dict:
    t0 = time.perf_counter()
    response = llm.invoke([
        SystemMessage(content="You are an HR specialist. Answer HR questions professionally."),
        *state["messages"]
    ])
    return {
        "messages": [response],
        "response": response.content,
        "metrics": {"hr_agent_calls": 1, "hr_agent_ms": round((time.perf_counter()-t0)*1000, 1)},
    }

def denied_node(state: EnterpriseState) -> dict:
    msg = AIMessage(content=f"Access denied: {state['error']}")
    return {"messages": [msg], "response": msg.content}

def budget_node(state: EnterpriseState) -> dict:
    msg = AIMessage(content=f"Request rejected: {state['error']}")
    return {"messages": [msg], "response": msg.content}

print("Specialist agents ready")

## Step 5 — Build the Full Enterprise Graph

In [ ]:
def route_after_auth(state: EnterpriseState) -> str:
    return state["next_agent"] if state["next_agent"] in {"db_agent", "report_agent", "hr_agent"} else "denied"

def route_after_cost(state: EnterpriseState) -> str:
    return state["next_agent"] if state["next_agent"] != "budget_exceeded" else "budget_exceeded"

checkpointer = MemorySaver()
builder = StateGraph(EnterpriseState)

builder.add_node("auth_gate",    auth_gate_node)
builder.add_node("cost_governor", cost_governor_node)
builder.add_node("db_agent",     db_agent_node)
builder.add_node("report_agent", report_agent_node)
builder.add_node("hr_agent",     hr_agent_node)
builder.add_node("denied",       denied_node)
builder.add_node("budget_exceeded", budget_node)

builder.add_edge(START, "auth_gate")
builder.add_conditional_edges("auth_gate", route_after_auth, {
    "db_agent": "cost_governor", "report_agent": "cost_governor",
    "hr_agent": "cost_governor", "denied": "denied",
})
builder.add_conditional_edges("cost_governor", route_after_cost, {
    "db_agent": "db_agent", "report_agent": "report_agent",
    "hr_agent": "hr_agent", "budget_exceeded": "budget_exceeded",
})
for node in ["db_agent", "report_agent", "hr_agent", "denied", "budget_exceeded"]:
    builder.add_edge(node, END)

graph = builder.compile(checkpointer=checkpointer)
print("Enterprise capstone graph compiled!")

## Step 6 — End-to-End Test

In [ ]:
def make_token(user_id: str, tenant_id: str, role: str) -> str:
    payload = {"sub": user_id, "tenant_id": tenant_id, "role": role,
               "exp": int(time.time()) + 3600}
    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")

def enterprise_invoke(token: str, question: str, token_budget: int = 2000):
    p = jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
    trace_id = str(uuid.uuid4())[:8]
    tid = f"{p['tenant_id']}:{p['sub']}"
    config = {"configurable": {"thread_id": tid}}
    
    print(f"\n{'='*65}")
    print(f"user={p['sub']} role={p['role']} trace={trace_id}")
    print(f"Q: {question}")
    
    result = graph.invoke({
        "messages": [HumanMessage(content=question)],
        "user_id": p["sub"], "tenant_id": p["tenant_id"], "role": p["role"],
        "next_agent": "", "trace_id": trace_id,
        "metrics": {}, "token_budget": token_budget, "tokens_used": 0,
        "response": "", "error": "",
    }, config=config)
    
    print(f"Route: {result.get('next_agent', '?')}")
    print(f"A: {result['response'][:200]}")
    print(f"Metrics: {result.get('metrics', {})}")

# Test scenarios
tokens = {
    "alice_user":    make_token("alice",  "acme", "user"),
    "bob_manager":   make_token("bob",    "acme", "manager"),
    "carol_admin":   make_token("carol",  "acme", "admin"),
}

enterprise_invoke(tokens["alice_user"],  "What are the key database metrics?")   # db_agent
enterprise_invoke(tokens["alice_user"],  "Generate a weekly report")             # denied (user can't access report_agent)
enterprise_invoke(tokens["bob_manager"], "Generate a weekly report")             # report_agent
enterprise_invoke(tokens["carol_admin"], "What is the HR policy on remote work?") # hr_agent

## Key Takeaways

| Pillar | Node | Pattern |
|--------|------|---------|
| Auth + RBAC | `auth_gate_node` | JWT decode → role check → route or deny |
| Cost control | `cost_governor_node` | Token estimate → budget check |
| Multi-tenant | thread_id | `f"{tenant_id}:{user_id}"` |
| Metrics | `Annotated[dict, merge_metrics]` | Accumulate across all nodes |
| Specialist routing | `add_conditional_edges` | `next_agent` field drives routing |

## 🏋️ Exercise
1. Add a `rate_limiter_node` between `auth_gate` and `cost_governor` — reject if same `user_id` made > 5 requests in 60s (use an in-memory dict with timestamps)
2. Add async support: convert all nodes to `async def` and use `await graph.ainvoke(...)`
3. Add an `audit_log: list` state field — append every request with timestamp, user, role, agent, latency

In [ ]:
# Your exercise solution here
